# SmolBench on Amazon Bedrock: a three-model ladder trio

This notebook runs both SmolBench studies on Amazon Bedrock, with reasoning switched on for
every model:

- **Periodic induction**: arms `intens`, `noise_intens` and `extens`, seeds 0–29, 9 count
  questions per replicate.
- **Horn-rule deduction**: arms `lem`, `pad`, `disc` and `both`, seeds 100–199, 3 replicates,
  temperature 0.7, each model at its ICLR chain length `m`.

The three models are all in the ICLR ladder, come from three different labs, and are
served by Bedrock:

| roster key | Bedrock id | lab | parameters | Horn `m` |
|---|---|---|---|---|
| `gemma-4-31b` | `google.gemma-4-31b` (served under `/openai/v1`) | Google | 30.7B dense | 64 |
| `nemotron-3-super-120b-a12b` | `nvidia.nemotron-super-3-120b` | NVIDIA | 120B MoE, 12B active | 20 |
| `deepseek-v3.1` | resolved from `GET /models` | DeepSeek | 685B MoE, 37B active | 64 |

The steps between them are 3.9x and 5.7x, or 22x end to end, in total parameters. Bedrock
hosts no other three ladder models from different large labs. Qwen3.5 is only on
SageMaker, and Gemma-4 E2B/12B and Nemotron-3 Nano-4B are not hosted.

**Code reused.** Requests go through `smolbench.evals.providers.aws`, the OpenAI-compatible
bedrock-mantle client. The induction quizzes come from
`notebooks/induction/run_study.make_quizzes`, and the runs use `ReplicateHarness`. The
Horn rungs come from `smolbench.deduction.horn.repro.render_rung` and are checked against
the ICLR digests; each cell is run and scored by `scripts/deduction/horn/sweep.run_cell`.
`bedrock_trio.py` adds only the roster, a per-model endpoint switch, a results store in
the released-results layout, and the reasoning checks.

**Differences from the ICLR runs:**

- **Output caps.** Bedrock caps output, reasoning included, at 8K tokens for DeepSeek V3.1
  and 32K for Nemotron Super; Gemma 4 has no documented cap, so it is sent 32K. The ICLR
  budgets were 131K (Horn) and at least 48K (induction). A truncated answer scores
  `length` (Horn) or `null`, counted as wrong (induction). Both rates are reported below.
  The published DeepSeek and Nemotron Horn rows came from Bedrock and were capped the
  same way.
- **Induction temperature.** It is not sent, as at ICLR, but the server default here is
  Bedrock's, not the vLLM `generation_config.json` default the ICLR runs used.
- **Seeding.** Every request here is seeded: the replicate seed for induction and the
  replicate index for Horn. The published DeepSeek and Nemotron Horn rows came from
  unseeded Bedrock Converse calls; the Gemma rows and all published induction rows came
  from self-hosted vLLM.

**Running it.**
1. Refresh your AWS credentials.
2. Set `SMOKE` in the next cell. `True` runs a few seeds per model; `False` runs the full
   protocol.
3. Run all cells.

A rerun resumes: finished replicates and rows are skipped. Results go to
`rebuttal/results/` (in the repo); rendered rungs go to `rebuttal/rungs/` (not committed,
since they are deterministic and digest-checked).

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rebuttal" / "bedrock_trio.py").exists())
sys.path.insert(0, str(REPO_ROOT / "rebuttal"))

import bedrock_trio as bt
from smolbench.deduction.horn import repro as horn_repro
from smolbench.induction import repro as induction_repro

SMOKE = True
REGION = "us-east-1"  # bedrock-mantle is verified here
MODELS = list(bt.TRIO)  # roster keys, smallest first
CONCURRENCY = 8  # parallel requests per model

if SMOKE:
    INDUCTION_SEEDS, HORN_SEEDS, HORN_REPS = [0, 1], [100, 101], 1
else:
    INDUCTION_SEEDS, HORN_SEEDS, HORN_REPS = list(range(30)), list(range(100, 200)), 3

print(f"SMOKE={SMOKE}: induction seeds {INDUCTION_SEEDS[0]}-{INDUCTION_SEEDS[-1]}, "
      f"Horn seeds {HORN_SEEDS[0]}-{HORN_SEEDS[-1]} x {HORN_REPS} reps, results in {bt.RESULTS_DIR}")

## Authenticate and resolve the model ids

This mints a short-term Bedrock bearer token from your AWS profile. An
`AWS_BEARER_TOKEN_BEDROCK` already set in the environment is used as is. The listing
combines both mantle paths, because Gemma 4 is only served under `/openai/v1`.

In [ ]:
bt.mint_token(REGION)
listing = set()
for path in ("/v1", "/openai/v1"):
    try:
        listing |= set(bt.list_mantle_models(REGION, path))
    except Exception as err:  # one path failing is reported, not fatal
        print(f"GET {path}/models failed: {err}")
TRIO = bt.resolve_ids(sorted(listing), {k: bt.TRIO[k] for k in MODELS})
for m in TRIO.values():
    print(f"{m.key:28s} {m.bedrock_id:34s} {bt.base_url(m, REGION)}")

## Force reasoning

For each model, this sends a one-line question with each of the model's reasoning fields
in turn and keeps the first one that returns reasoning. If none does, the cell fails, and
the model is not run without reasoning. The probes are saved to `results/probes.json`.

In [ ]:
REASONING, PROBES = {}, {}
for key, model in TRIO.items():
    REASONING[key], PROBES[key] = bt.pick_reasoning_args(model, REGION)
    print(f"{key:28s} {REASONING[key]}  <- {PROBES[key][-1]}")
bt.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(bt.RESULTS_DIR / "probes.json").write_text(json.dumps(
    {"region": REGION, "ids": {k: m.bedrock_id for k, m in TRIO.items()},
     "reasoning_args": REASONING, "probes": PROBES}, indent=2) + "\n")

## Check the induction prompts against the released ones

For seed 0 of each model, the regenerated prompts of all three arms, including the
`noise_intens` prompts padded to a token count under each model's own tokenizer, must
equal the prompts in the released ICLR results byte for byte. The released files are read
anonymously from the public bucket. The tokenizers come from Hugging Face at the pinned
revisions, so the gated `google/gemma-4-31B-it` needs `HF_TOKEN` unless it is already
cached.

In [ ]:
run_study = bt.load_run_study()
for key in TRIO:
    mine = run_study.make_quizzes(0, key)
    released = bt.released_queries(key, 0)
    for arm in bt.INDUCTION_ARMS:
        assert released.get(arm), f"{key}/{arm}: released file has no prompts"
        assert [q.prompt for q in mine[arm]] == released[arm], f"{key}/{arm}: prompts differ"
    print(f"{key}: 27 prompts identical to the released ICLR prompts")

## Run the induction study

The three arms of one seed are sent together, and one YAML per replicate is written to
`results/induction/induction/<model>/seed=<seed>/<arm>--<timestamp>.yaml`, the
released-results layout.

In [ ]:
harness = bt.induction_harness(list(TRIO.values()), INDUCTION_SEEDS)
for key, model in TRIO.items():
    print(f"== {key}")
    bt.run_induction(harness, model, REGION, REASONING[key], max_parallel=CONCURRENCY)

## Run the Horn study

Each model's rung is rendered and checked against the ICLR digests, then every
(arm, seed, replicate) cell is sent and scored with the ICLR answer extraction. Rows go to
`results/horn/<model>.jsonl`. A transport failure is written as an `exception` row and is
retried on the next run.

In [ ]:
for key, model in TRIO.items():
    m = bt.horn_m(key)
    rung = bt.render_horn_rung(m, HORN_SEEDS)
    rows = bt.run_horn(model, REGION, REASONING[key], rung, HORN_SEEDS, HORN_REPS, concurrency=CONCURRENCY)
    verdicts = {}
    for r in rows:
        verdicts[r["verdict"]] = verdicts.get(r["verdict"], 0) + 1
    print(f"== {key} m{m}: {len(rows)} new rows {verdicts}")

## Results

Accuracies and pass rates are shown next to the published ICLR values. The breakdowns
show how many answers were truncated or could not be graded, and how many responses came
back without reasoning, which must be zero.

In [ ]:
print(induction_repro.report(bt.RESULTS_DIR / "induction"))

In [ ]:
rows_files = [bt.horn_rows_path(k) for k in TRIO if bt.horn_rows_path(k).exists()]
print(horn_repro.report(rows_files))

In [ ]:
ind = bt.induction_reasoning_report()
horn = bt.horn_breakdown(rows_files)
print("induction (model, arm): counts")
for k, v in sorted(ind.items()):
    print(f"  {k}: {v}")
print("horn (model, arm): counts")
for k, v in sorted(horn.items()):
    print(f"  {k}: {v}")
unreasoned = {k: v["unreasoned"] for k, v in {**ind, **horn}.items() if v.get("unreasoned")}
assert not unreasoned, f"responses without reasoning: {unreasoned}"